# HPO for Embedding Downstream Segmentation (Burn Scars)

This notebook runs **hyperparameter optimisation (HPO)** over the learning rate for the burn scars downstream segmentation pipeline using:

| Component | Role |
|-----------|------|
| **TerraTorch CLI** | `terratorch fit -c downstream_segmentation_burnscars.yaml` – trains the FCN decoder on top of TerraMind embeddings |
| **jsonargparse overrides** | dotted CLI flags (e.g. `--optimizer.init_args.lr 3e-4`) let iterate inject sampled values at runtime without modifying the base YAML |
| **iterate** | orchestrates Optuna trials, sets `ITERATE_PARAM_*` env-vars for every trial, collects the metric written to `ITERATE_OUT_FILE` |

## Files used alongside this notebook

| File | Purpose |
|------|---------|
| `hpo_burnscars.yaml` | iterate HPO config – defines the `lr` search space and the metric to minimise (`val_loss`) |
| `train_burnscars_trial.py` | trial script called by iterate for each Optuna trial |
| `downstream_segmentation_burnscars.yaml` | base TerraTorch config (unchanged) |

## Prerequisites

Run all cells from the top — cell 1 installs required packages.


## 1. Install Dependencies

In [ ]:
!pip install -qq terratorch==1.2.4 optuna optuna-dashboard terratorch-iterate==0.4



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## 2. Download Data and Embeddings

The cells below **lazily** download the HLS Burn Scars dataset archive, extract it, and fetch pre-computed TerraMind embeddings from Google Drive — only if the data is not already present locally.

In [ ]:
# Download and extract the HLS Burn Scars dataset if not already present
import tarfile
from pathlib import Path

try:
    import gdown
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gdown", "-q"])
    import gdown

dataset_path = Path("hls_burn_scars")
archive_path = dataset_path.with_suffix(".tar.gz")

# Download archive if it does not exist
if not archive_path.is_file() and not dataset_path.is_dir():
    print("Downloading dataset archive...")
    gdown.download("https://drive.google.com/uc?id=1yFDNlGqGPxkc9lh9l1O70TuejXAQYYtC", output=str(archive_path), quiet=False)
    print(f"Downloaded {archive_path}")
elif archive_path.is_file():
    print(f"Archive already present: {archive_path}")

# Extract archive if dataset directory does not exist
if not dataset_path.is_dir():
    print("Extracting dataset archive...")
    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(".")
    print(f"Dataset extracted to {dataset_path}")
else:
    print(f"Dataset already present at {dataset_path}")

# Download split files if not present
import urllib.request
splits_dir = dataset_path / "splits"
splits_base_url = "https://raw.githubusercontent.com/IBM/peft-geofm/main/datasets_splits/burn_scars"
splits_dir.mkdir(parents=True, exist_ok=True)
for fname in ["train.txt", "val.txt", "test.txt"]:
    dest = splits_dir / fname
    if not dest.exists():
        print(f"Downloading split file: {fname}...")
        urllib.request.urlretrieve(f"{splits_base_url}/{fname}", dest)
    else:
        print(f"Split file already present: {dest}")

# Download pre-computed TerraMind embeddings if not already present
import zipfile, tempfile, shutil

embedding_path = dataset_path / "embeddings_terramind" / "layer_00"
zip_path = Path("hls_burn_scars_embeddings_terramind.zip")
gdrive_file_id = "1SA-WVWVC0d-s5BRKrup54lp7oFmjbSkR"

if not embedding_path.is_dir() or not any(embedding_path.glob("*_embedding.tif")):
    print("Embeddings not found. Checking for zip file...")
    if not zip_path.exists():
        print("Zip file not found. Downloading from Google Drive...")
        gdown.download(id=gdrive_file_id, output=str(zip_path), quiet=False)
        print(f"Downloaded {zip_path}")
    else:
        print(f"Zip file already present: {zip_path}")

    print("Extracting zip file...")
    with tempfile.TemporaryDirectory() as tmpdir:
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(tmpdir)
        layer_dirs = list(Path(tmpdir).rglob("layer_00"))
        if not layer_dirs:
            raise RuntimeError("Could not find 'layer_00' directory inside the zip file.")
        src_layer = layer_dirs[0]
        embedding_path.parent.mkdir(parents=True, exist_ok=True)
        if embedding_path.exists():
            shutil.rmtree(embedding_path)
        shutil.copytree(src_layer, embedding_path)

    extracted_count = len(list(embedding_path.glob("*_embedding.tif")))
    print(f"Embeddings extracted to {embedding_path} ({extracted_count} files)")
else:
    print(f"Embeddings already present at {embedding_path} ({len(list(embedding_path.glob('*_embedding.tif')))} files)")

In [ ]:
# Validate that the required data and embeddings are present
root = Path("hls_burn_scars")
dataset_path = root / "data"
embedding_path = root / "embeddings_terramind" / "layer_00"

ok = True
msgs = []

if not dataset_path.is_dir():
    ok = False
    msgs.append(f"- Dataset path not found: {dataset_path}")

if not embedding_path.is_dir():
    ok = False
    msgs.append(f"- Embedding path not found: {embedding_path}")

dataset_files = list(dataset_path.glob("*_merged.tif")) if dataset_path.is_dir() else []
embedding_files = list(embedding_path.glob("*_embedding.tif")) if embedding_path.is_dir() else []

if len(dataset_files) not in (804, 1608):
    ok = False
    msgs.append(f"- Expected 804 or 1608 dataset files (*_merged.tif), found {len(dataset_files)}")

if len(embedding_files) not in (804, 1608):
    ok = False
    msgs.append(f"- Expected 804 or 1608 embedding files (*_embedding.tif), found {len(embedding_files)}")

if ok:
    print(f"Data available. ({len(dataset_files)} dataset files, {len(embedding_files)} embedding files)")
else:
    raise RuntimeError(
        "Required data not found or incomplete.\n"
        + "\n".join(msgs)
        + "\n\nPlease check the paths and re-run the download cell above."
    )

## 3. Inspect the iterate HPO Config

The file `hpo_burnscars.yaml` defines **three sections** interpreted by iterate:

* **`metrics`** – the metric name that `train_burnscars_trial.py` writes to `ITERATE_OUT_FILE`.  
  Here: `val_loss` (lower is better).
* **`static`** – fixed key/value pairs forwarded to every trial as `ITERATE_PARAM_<KEY>` env-vars.  
  Here: the base config file name and the number of training epochs per trial.
* **`hpo`** – parameters Optuna will sample. Each entry specifies `type`, `low`, `high`, and optionally `log`.  
  Here: `lr` is sampled log-uniformly in $[10^{-5},\ 10^{-2}]$.

In [ ]:
# Display the iterate HPO config
with open("hpo_burnscars.yaml") as f:
    print(f.read())

## 4. Inspect the Trial Script

`train_burnscars_trial.py` is the script iterate calls for every Optuna trial.

**Key steps inside the script:**

1. Read parameters from `ITERATE_PARAM_*` environment variables — this is how `_iterate2.py` delivers sampled and static values to the trial script:
   ```
   ITERATE_PARAM_LR=<lr>
   ITERATE_PARAM_CONFIG=<config>
   ITERATE_PARAM_EPOCHS=<epochs>
   ```
2. Call the **TerraTorch CLI** with a [jsonargparse](https://jsonargparse.readthedocs.io/) dotted-key override:
   ```bash
   terratorch fit -c downstream_segmentation_burnscars.yaml \
       --optimizer.init_args.lr <lr>        # <-- jsonargparse override
       --trainer.max_epochs <epochs>
       --trainer.default_root_dir hpo_trial_<n>
   ```
   Because TerraTorch's CLI is built on Lightning-CLI / jsonargparse, **any nested config key** can be overridden via a dotted flag without touching the YAML.
3. Parse Lightning's `metrics.csv` (written automatically inside the trial log dir) to extract `val/loss`.
4. Write `val_loss: <value>` to `ITERATE_OUT_FILE` so iterate can pass it to Optuna.

In [ ]:
# Display the trial script
with open("train_burnscars_trial.py") as f:
    print(f.read())

## 5. Run the HPO Loop with iterate

The cell below launches the iterate HPO loop.  
iterate will:
1. Create an Optuna study stored in `iterate_burnscars.db`.
2. Sample a learning rate from the log-uniform range defined in `hpo_burnscars.yaml`.
3. Set `ITERATE_PARAM_LR`, `ITERATE_PARAM_CONFIG`, and `ITERATE_PARAM_EPOCHS` env-vars and call `train_burnscars_trial.py`.
4. Read `val_loss` from `ITERATE_OUT_FILE` and tell Optuna the result.
5. Repeat for `--optuna-n-trials` trials.

> **Tip:** increase `--optuna-n-trials` for a more thorough search.

### Expected warnings in the trial output (all harmless)

You will see lines like the following — they can safely be ignored:

```
[W:onnxruntime:Default, device_discovery.cc:211] GPU device discovery failed …
    ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
```
**Cause:** onnxruntime scans for GPU devices at import time via Linux sysfs. If the sysfs GPU path doesn't exist (e.g. on CPU-only instances or SageMaker), it logs this warning but continues normally.

```
E … cuda_fft.cc:467] Unable to register cuFFT factory: … already registered
E … cuda_dnn.cc:8579] Unable to register cuDNN factory: … already registered
E … cuda_blas.cc:1407] Unable to register cuBLAS factory: … already registered
W … computation_placer.cc:177] computation placer already registered.
```
**Cause:** TensorFlow/JAX CUDA plugins attempt to register themselves when multiple CUDA-aware packages are imported in the same process. The "already registered" message means another package got there first — harmless duplicate registration noise.

Neither set of warnings affects training correctness or GPU utilisation.


In [ ]:
import sys, os
from pathlib import Path

# Resolve the directory where this notebook's sibling files live
NB_DIR = Path(os.getcwd())
if not (NB_DIR / "train_burnscars_trial.py").exists():
    NB_DIR = NB_DIR / "examples" / "embeddings"

SCRIPT    = NB_DIR / "train_burnscars_trial.py"
HPO_YAML  = NB_DIR / "hpo_burnscars.yaml"
DB_PATH   = NB_DIR / "iterate_burnscars.db"

# Absolute path to the iterate binary in the same venv as this kernel
ITERATE   = Path(sys.executable).parent / "iterate"

print(f"iterate   : {ITERATE}  (exists={ITERATE.exists()})")
print(f"script    : {SCRIPT}   (exists={SCRIPT.exists()})")
print(f"hpo yaml  : {HPO_YAML} (exists={HPO_YAML.exists()})")

!chmod 755 {SCRIPT}

!{ITERATE} \
    --script {SCRIPT} \
    --optuna-study-name burnscars_hpo \
    --optuna-db-path "sqlite:///{DB_PATH}" \
    --hpo-yaml {HPO_YAML} \
    --optuna-n-trials 5


## 6. Inspect Optuna Results

After the loop finishes, we load the Optuna study from the SQLite database and display the best trial.

In [ ]:
import optuna, sys, os
from pathlib import Path

NB_DIR = Path(os.getcwd())
if not (NB_DIR / "iterate_burnscars.db").exists():
    NB_DIR = NB_DIR / "examples" / "embeddings"
DB_PATH = NB_DIR / "iterate_burnscars.db"

study = optuna.load_study(
    study_name="burnscars_hpo",
    storage=f"sqlite:///{DB_PATH}",
)

print(f"Number of finished trials: {len(study.trials)}")
best = study.best_trial
print(f"\nBest trial:")
print(f"  val_loss : {best.value:.6f}")
print(f"  lr       : {best.params['lr']:.2e}")


In [ ]:
# Plot optimisation history and parameter importance
import matplotlib.pyplot as plt
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
)

ax1 = plot_optimization_history(study)
ax1.set_title("Optimisation History")
ax1.figure.tight_layout()
plt.show()

ax2 = plot_param_importances(study)
ax2.set_title("Parameter Importances")
ax2.figure.tight_layout()
plt.show()


## 7. (Optional) Launch Optuna Dashboard

For an interactive view of all trials, run the following in a terminal:

```bash
optuna-dashboard --host 0.0.0.0 --port 8080 sqlite:///iterate_burnscars.db
```

or directly from the notebook:

In [ ]:
# Uncomment to launch the dashboard (blocks the notebook while running)
# !optuna-dashboard --host 0.0.0.0 --port 8080 sqlite:///iterate_burnscars.db

## 8. Re-train with Best Learning Rate

Use the best learning rate found by iterate/Optuna to run a full training with more epochs.

Because TerraTorch uses jsonargparse, we can pass the override directly on the CLI without modifying `downstream_segmentation_burnscars.yaml`.

In [ ]:
import optuna, sys, os
from pathlib import Path

NB_DIR = Path(os.getcwd())
if not (NB_DIR / "iterate_burnscars.db").exists():
    NB_DIR = NB_DIR / "examples" / "embeddings"
DB_PATH   = NB_DIR / "iterate_burnscars.db"
CONFIG    = NB_DIR / "downstream_segmentation_burnscars.yaml"
TERRATORCH = Path(sys.executable).parent / "terratorch"

study = optuna.load_study(study_name="burnscars_hpo", storage=f"sqlite:///{DB_PATH}")
best_lr = study.best_trial.params["lr"]
print(f"Best LR: {best_lr:.2e}")

!{TERRATORCH} fit \
    -c {CONFIG} \
    --optimizer.init_args.lr {best_lr} \
    --trainer.max_epochs 50
